# 01 · Stage (a) scan-quality classifier (TinyCNN4) · fast.ai on a Kaggle T4

**Goal.** Train the A1 `TinyCNN4` scan-quality classifier from the Cycle-2 shootout on the
pre-generated synthetic cache, pick the no-pulse operating point on val, then export a static
INT8 ONNX for the phone.

* Classes: `good`, `low_snr`, `motion`, `exposure_event`, `no_pulse`.
* Input: `(3, 80, 128)` composite: log spectrograms (top 64 rows) + a 16-patch mini-STMap
  (bottom 16 rows), built by `mp_data.make_sqa_image`.
* Kaggle settings: **GPU T4 x1**, internet on for the pinned install, and the cache dataset
  attached (see §1). Expect about a minute of training for 12 epochs on a T4.

Not a medical device. Synthetic data only; swap in MCD-rPPG / MMPD with subject-disjoint splits
next cycle (same dict keys).

## Shootout §6 lessons applied here

| # | Lesson (FastAI-CV-Shootout-Report §6) | Where in this notebook |
|---|---|---|
| L1 | Pre-generate the data cache once (the 9,900-session cache is ~1.3 GB float16 `.npz`) and attach it as a Kaggle dataset; never regenerate inside a GPU session | §1 Data |
| L2 | Keep the float16 tensors resident on the GPU, batch 256 with the LR scaled up, `num_workers=0` | §2 DataLoaders |
| L3 | No `vision_learner`, no `aug_transforms`, no flips or crops: the axes are time and frequency, not pixels | §2 DataLoaders, §3 Model |
| L4 | `to_fp16()` mixed precision, but keep spectral maths (DFT / power / log) in FP32 with `torch.autocast(enabled=False)` | §3 Model |
| L5 | `pretrained=False` everywhere: ImageNet weights do not transfer to spectrograms or STMaps | §3 Model |
| L6 | Pin versions (torch 2.15.0, fastai 2.8.12, timm 1.0.31, onnx 1.23.2, onnxruntime 1.29.0, Python 3.13) and export with `torch.onnx.export(dynamo=False, opset_version=17)` | §0 Setup, §5 Export |
| L7 | Budget epochs from measured speed (C3: ~2 min/epoch on CPU, < 10 s/epoch on a T4); static QDQ INT8 for convnets, never dynamic INT8 (5-30x slower on x86) | §4 Train, §5 Export |

## 0 · Setup (L6)

In [ ]:
# L6: pin versions. Kaggle images move under you; a pinned install makes reruns comparable.
!pip install -q torch==2.15.0 fastai==2.8.12 timm==1.0.31 onnx==1.23.2 onnxruntime==1.29.0

In [ ]:
import os, sys, time, json, math, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./out")
OUT.mkdir(parents=True, exist_ok=True)
print(sys.version.split()[0], torch.__version__, DEVICE,
      torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU")
# SMOKE=1 runs every cell end to end on tiny random tensors (CPU is fine); use it before a GPU run.
SMOKE = os.environ.get("SMOKE", "0") == "1"

## 1 · Data (L1)

In [ ]:
# L1: the cache is built ONCE (mp_data.build_split, ~1.3 GB float16 for all splits) and attached
# as a Kaggle dataset. Never regenerate it inside a GPU session.
SQA_CLASSES = ["good", "low_snr", "motion", "exposure_event", "no_pulse"]
NP_IDX = SQA_CLASSES.index("no_pulse")
CACHE = Path(os.environ.get("PRANASCAN_CACHE", "/kaggle/input/pranascan-mp-cache"))
FILES = {"train": "train_600.npz", "val": "val_100.npz", "holdout": "holdout_150.npz"}


def load(split):
    if SMOKE:
        rng = np.random.default_rng({"train": 0, "val": 1, "holdout": 2}[split])
        n = 320 if split == "train" else 160
        return {"sqa": rng.standard_normal((n, 3, 80, 128)).astype(np.float16),
                "sqa_label": rng.integers(0, 5, n)}
    with np.load(CACHE / FILES[split], allow_pickle=False) as z:
        return {"sqa": z["sqa"].astype(np.float16), "sqa_label": z["sqa_label"].astype(np.int64)}


tr, va, ho = load("train"), load("val"), load("holdout")
for k, d in (("train", tr), ("val", va), ("holdout", ho)):
    print(k, d["sqa"].shape, d["sqa"].dtype, np.bincount(d["sqa_label"], minlength=5))

## 2 · DataLoaders (L2, L3)

In [ ]:
def gpu_dls(xtr, ytr, xva, yva, bs=256):
    """L2: float16 tensors resident on the GPU, no worker processes, no batch transforms (L3)."""
    from fastai.data.core import DataLoaders
    from torch.utils.data import TensorDataset

    to = lambda a, dt: torch.as_tensor(a).to(DEVICE, dt)
    ds_tr = TensorDataset(to(xtr, torch.float16), to(ytr, ytr_dtype(ytr)))
    ds_va = TensorDataset(to(xva, torch.float16), to(yva, ytr_dtype(yva)))
    return DataLoaders.from_dsets(ds_tr, ds_va, bs=bs, num_workers=0, pin_memory=False,
                                  device=DEVICE)


def ytr_dtype(y):
    return torch.long if np.asarray(y).dtype.kind in "iu" else torch.float32


class Fp32Input(nn.Module):
    """Cast the float16 batch up once; to_fp16 then autocasts the convs back to half."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return self.m(x.float())


# L2: the reference recipe used bs=64, lr=3e-3; at bs=256 scale the LR by sqrt(256/64) = 2.
BS, LR = 256, 3e-3 * math.sqrt(256 / 64)

## 3 · Model (L3, L5)

In [ ]:
class TinyCNN4(nn.Module):
    """A1 from the shootout: 4 x (conv-BN-ReLU-maxpool), global mean, linear. ~46k params."""

    def __init__(self, in_ch=3, n_classes=5, widths=(16, 32, 48, 64)):
        super().__init__()
        layers, c = [], in_ch
        for w in widths:
            layers += [nn.Conv2d(c, w, 3, padding=1, bias=False), nn.BatchNorm2d(w),
                       nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c = w
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(c, n_classes)

    def forward(self, x):
        return self.head(self.features(x).mean((2, 3)))


def build(arch):
    if arch == "A1_tinycnn4":
        return TinyCNN4()
    if arch == "A2_mobilenetv3_small":
        import timm
        # L5: pretrained=False. ImageNet features do not transfer to spectrogram/STMap axes.
        return timm.create_model("mobilenetv3_small_100", pretrained=False, in_chans=3,
                                 num_classes=5)
    raise KeyError(arch)


ARCH = "A1_tinycnn4"  # try "A2_mobilenetv3_small" for the comparison row
print(ARCH, sum(p.numel() for p in build(ARCH).parameters()), "params")

## 4 · Train (L4, L7)

In [ ]:
from fastai.callback.schedule import fit_one_cycle  # noqa: F401  (patches Learner.fit_one_cycle)
from fastai.learner import Learner
from fastai.losses import CrossEntropyLossFlat
from fastai.metrics import accuracy

counts = np.bincount(tr["sqa_label"], minlength=5)
weight = torch.tensor(counts.sum() / (5 * np.maximum(counts, 1)), dtype=torch.float32).to(DEVICE)
dls = gpu_dls(tr["sqa"], tr["sqa_label"], va["sqa"], va["sqa_label"], bs=BS)
torch.manual_seed(SEED)
model = Fp32Input(build(ARCH))
learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(weight=weight), metrics=accuracy)
if DEVICE.type == "cuda":
    learn = learn.to_fp16()  # L4: mixed precision on the T4's tensor cores
EPOCHS = 2 if SMOKE else 12   # L7: ~5 s/epoch on a T4 for A1 at bs 256
t0 = time.time()
learn.fit_one_cycle(EPOCHS, LR)
print(f"trained {EPOCHS} epochs in {time.time() - t0:.0f} s")
net = learn.model.m.float().eval()
torch.save(net.state_dict(), OUT / f"{ARCH}.pt")

## 4b · Evaluate and pick the no-pulse operating point on val

In [ ]:
from sklearn.metrics import f1_score, recall_score


def probs_of(m, x, bs=512):
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(x), bs):
            xb = torch.as_tensor(x[i:i + bs]).to(DEVICE).float()
            out.append(torch.softmax(m.to(DEVICE)(xb), 1).cpu().numpy())
    return np.concatenate(out)


def with_theta(p, theta):
    """Predict no_pulse whenever p(no_pulse) >= theta, else argmax of the other four."""
    q = p.copy(); q[:, NP_IDX] = -1
    pred = q.argmax(1); pred[p[:, NP_IDX] >= theta] = NP_IDX
    return pred


def report(pred, y):
    rec = recall_score(y, pred, labels=range(5), average=None, zero_division=0)
    good = SQA_CLASSES.index("good")
    return {"macro_f1": round(float(f1_score(y, pred, labels=range(5), average="macro")), 3),
            "no_pulse_recall": round(float(rec[NP_IDX]), 3),
            "false_abstain_good": round(float(np.mean(pred[y == good] != good)), 3)}


pva, pho = probs_of(net, va["sqa"]), probs_of(net, ho["sqa"])
# Operating point chosen on VAL only: smallest theta that flags >= 97 % of val no-pulse scans.
pn = np.sort(pva[va["sqa_label"] == NP_IDX, NP_IDX])
THETA = float(pn[int(np.floor(0.03 * len(pn)))]) if len(pn) else 0.5
print("argmax  holdout:", report(pho.argmax(1), ho["sqa_label"]))
print(f"theta={THETA:.3f} holdout:", report(with_theta(pho, THETA), ho["sqa_label"]))
json.dump({"arch": ARCH, "theta_no_pulse": THETA}, open(OUT / f"{ARCH}_op.json", "w"))

## 5 · Export: FP32 + static INT8 QDQ ONNX (L6, L7)

In [ ]:
def export_onnx(model, shape, path):
    """L6: TorchScript exporter (dynamo=False), opset 17, dynamic batch axis."""
    model = model.float().eval().cpu()
    torch.onnx.export(model, (torch.randn(*shape),), str(path), dynamo=False, opset_version=17,
                      input_names=["x"], output_names=["logits"],
                      dynamic_axes={"x": {0: "batch"}, "logits": {0: "batch"}})
    return path


def quantize_qdq(fp32_path, out_path, calib, op_types=None):
    """L7: static INT8 QDQ (per-channel weights). Dynamic INT8 is 5-30x slower on x86 CPUs."""
    from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType,
                                          quantize_static)
    from onnxruntime.quantization.shape_inference import quant_pre_process

    class Reader(CalibrationDataReader):
        def __init__(self, x):
            self.it = iter([{"x": x[i:i + 1].astype(np.float32)} for i in range(len(x))])

        def get_next(self):
            return next(self.it, None)

    pre = str(out_path).replace(".onnx", "_pre.onnx")
    quant_pre_process(str(fp32_path), pre, skip_symbolic_shape=True)
    kw = dict(quant_format=QuantFormat.QDQ, per_channel=True,
              activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
    if op_types:
        kw["op_types_to_quantize"] = op_types
    quantize_static(pre, str(out_path), Reader(calib), **kw)
    os.remove(pre)
    return out_path


def ort_logits(path, x, bs=128):
    import onnxruntime as ort
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    return np.concatenate([s.run(None, {"x": x[i:i + bs].astype(np.float32)})[0]
                           for i in range(0, len(x), bs)])


def cpu1_latency_ms(path, shape, warm=30, runs=200):
    import onnxruntime as ort
    so = ort.SessionOptions(); so.intra_op_num_threads = 1; so.inter_op_num_threads = 1
    s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    x = np.random.default_rng(0).standard_normal(shape).astype(np.float32)
    for _ in range(warm):
        s.run(None, {"x": x})
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); s.run(None, {"x": x}); ts.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(ts))

In [ ]:
SHAPE = (1, 3, 80, 128)
fp32 = export_onnx(net, SHAPE, OUT / f"{ARCH}_fp32.onnx")
calib = va["sqa"][:: max(1, len(va["sqa"]) // 256)].astype(np.float32)
int8 = quantize_qdq(fp32, OUT / f"{ARCH}_int8_static.onnx", calib)
for p in (fp32, int8):
    lo = ort_logits(p, ho["sqa"][:512].astype(np.float32))
    pr = np.exp(lo - lo.max(1, keepdims=True)); pr /= pr.sum(1, keepdims=True)
    print(p.name, f"{os.path.getsize(p) / 1e6:.3f} MB",
          f"cpu1 {cpu1_latency_ms(p, SHAPE):.2f} ms",
          report(with_theta(pr, THETA), ho["sqa_label"][:512]))

## Next

* Download `A1_tinycnn4_int8_static.onnx` and `A1_tinycnn4_op.json` from the Output tab; the
  phone gate uses `theta_no_pulse` exactly as `with_theta` does here.
* Holdout is scored once. If you change anything, re-pick theta on val, not on holdout.
* Real-data swap: keep `load()`'s keys (`sqa`, `sqa_label`) and the subject-disjoint split rule.